### Code to prep the file for the tps-plot

(somewhat) expected format of information: names=["freq_ghz", "dur_ms", "Fluence"]) <br>

Width of burst is calculated as follows: width_t = burst.end_acf_range.values.max() - burst.begin_acf_range.values.min()

We take the beginning of the ACF (beginning of the first component) and the end of the last component, that is the width. 

In [1]:
import numpy as np
from matplotlib import rcParams
import pandas as pd
from tqdm import tqdm

In [2]:
def toa_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_bary_tdb_inf_freq"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_bary_tdb_inf_freq"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

In [3]:
def fluence_looper(df_fluence):
    """
    function to sum over the fluences components of bursts
    
    Input: The burst_csv file with all information, spc/sfxc/scale
    This is needed because we return a df with fluence (spc) and toa (sfxc)
    
    return: pandas dataframe with: id/fluence/telescope/c_freq/toa/width_ms
    """

    #the burst names to loop over
    exps = df_fluence['id'].unique()

    #Create a list with the telescope names for easy filtering later on
    exps_tel = [i.split('-')[1] for i in exps]

    #loop over every burst to sum the fluence
    info_list = []
    for exp, telescope in zip(exps, exps_tel):
        
        #Fluence: Making a small temp df for each exp, only taking the spc values
        if telescope != 'st':
            fluence_df_exp_temp = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'spc')]
            
            #TOA
            fluence_df_exp_temp_mjd = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'sfxc')]
            toa = toa_calulator(fluence_df_exp_temp_mjd) 
                
        elif telescope == 'st':
            fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
            #TOA
            toa = toa_calulator(fluence_df_exp_temp) 
            
        #Calculate the total fluence
        fluence = sum(fluence_df_exp_temp['fluence_jyms'].values)
        
        #width in ms
        width_t = fluence_df_exp_temp.end_acf_range.values.max() - fluence_df_exp_temp.begin_acf_range.values.min()

        #Since the F0 is the same for all components, take all components
        cent_freq = fluence_df_exp_temp['f0_MHz'].values[0]
                            
        #Add to the list
        info_list.append([fluence, cent_freq, toa, width_t])
        
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)

    #Convert the info to dataframe
    new_data = {'id': exps, 'fluence': info_list_arr[:,0], 'telescope': exps_tel, \
                'c_freq':info_list_arr[:,1], 'toa': info_list_arr[:,2],\
               'width_ms': info_list_arr[:,3]}
    
    df_new_fluence = pd.DataFrame(data=new_data)

    #add only the index number as a seperate column for potential later use
    df_new_fluence['burst-index'] = df_new_fluence['id'].str.split('-').str[0].str[1:].astype(int)
    
    return df_new_fluence

In [5]:
def load_df(file):
    """
    Function to load in the burst.csv file and return the Fluences, observed frequency and width per bursts
    
    band: specifiy wether Lband or Pband bursts
    
    notes:
    B84  is missing, since it was determined to not be real after classification
    B130 is actively removed from calculations, since the RFI made it impossible to measure properties
    """

    #load in the main csv file
    fluence_df = pd.read_csv(file, index_col=0)
    fluence_df.sort_values(by=['id'])

    # Specifically remove B130-tr; because it was not possible to calculate a fluence
    mask = fluence_df['id'] == 'B130-tr'
    fluence_df = fluence_df[~mask]

    ##Wb, On and Tr Fluences
    #Only take the spc fluences
    #df_spc = fluence_df[fluence_df['src'] == 'spc']
    df_combined = fluence_looper(fluence_df)    
    df_combined.sort_values(by=['id'])
      
        
    #Dropping the duplicate bursts:    
    #In case there is overlap with Torun, drop Torun due to bad baseline
    #In case St/Wb overlap, drop the fainter burst
    #29-Aug-23 double checked the overlaps
    #tr_overlap = ["B07-tr","B13-tr","B15-tr","B16-tr","B37-tr"]
    #st_overlap = ["B30-st","B53-st","B63-st","B64-st","B68-st","B69-st","B104-st","B107-st"]
    #wb_overlap = ["B82-wb","B85-wb","B106-wb"]
    
    #st_overlap = []
    #wb_overlap = []
    
    #Dropping the overlap bursts
    #overlaps = np.concatenate((tr_overlap, st_overlap, wb_overlap))
    #df_combined = df_combined[~df_combined['id'].isin(overlaps)]
                    
    #Resort and reset the df
    df_combined = df_combined.sort_values(by=['burst-index'])
    df_combined = df_combined.reset_index(drop=True)
    
    display(df_combined)
    name_csv = "tps_r117_europe.csv"
    df_combined.to_csv(name_csv, header=True, index=True, na_rep='NA')
    return df_combined

fluence_df = load_df("../dbs/burst_stats_r117.csv")
#display(fluence_df)

,id,fluence,telescope,c_freq,toa,width_ms,burst-index
0,B01-st,42.160639,st,1381.464844,59867.550262,9.611947,1
1,B02-st,24.260761,st,1381.464844,59868.720719,7.864320,2
2,B03-st,29.733580,st,1381.464844,59868.806998,13.544107,3
3,B04-st,11.796727,st,1381.464844,59868.952065,4.587520,4
4,B05-tr,40.749134,tr,1414.000000,59868.952070,3.968000,5
...,...,...,...,...,...,...,...
141,B127-wb,274.247565,wb,328.000000,59941.687390,20.480000,127
142,B128-wb,2629.400017,wb,328.000000,59941.730176,32.768000,128
143,B129-st,46.150610,st,1381.464844,59949.882698,12.670293,129
144,B131-st,375.073496,st,1381.464844,59978.532358,13.325653,131
